# OlegShifter Android — сборка APK (нативный Kotlin + Go-ядро)

Надёжная замена buildozer: компилируем Go-протокол через **gomobile** в `.aar`, затем собираем APK через **Gradle**.

Включает **VPN-режим** (весь трафик телефона через туннель, на gvisor/tun2socks).

Выполняй ячейки по порядку (Shift+Enter). В **Шаге 4** укажи свой GitHub-репозиторий. Полная сборка ~12–18 минут (gvisor компилируется небыстро).

## Шаг 1. Go 1.25 + JDK 17

In [ ]:
import os
# Go 1.25+ обязателен (golang.org/x/mobile = gomobile требует 1.25)
!wget -q https://go.dev/dl/go1.25.0.linux-amd64.tar.gz -O /tmp/go.tgz
!rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
!apt-get -qq update >/dev/null && apt-get -qq install -y openjdk-17-jdk unzip >/dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['GOPATH'] = '/root/go'
os.environ['GOTOOLCHAIN'] = 'auto'
os.environ['PATH'] = '/usr/local/go/bin:/root/go/bin:' + os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!go version && java -version

## Шаг 2. Android SDK + NDK

In [ ]:
import os
SDK = '/root/android-sdk'
!mkdir -p {SDK}/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmt.zip
!unzip -q -o /tmp/cmt.zip -d {SDK}/cmdline-tools
!mv -f {SDK}/cmdline-tools/cmdline-tools {SDK}/cmdline-tools/latest 2>/dev/null || true
os.environ['ANDROID_HOME'] = SDK
os.environ['ANDROID_SDK_ROOT'] = SDK
os.environ['PATH'] = f'{SDK}/cmdline-tools/latest/bin:{SDK}/platform-tools:' + os.environ['PATH']
!yes | sdkmanager --licenses >/dev/null 2>&1
!sdkmanager 'platform-tools' 'platforms;android-34' 'build-tools;34.0.0' 'ndk;25.2.9519653' >/dev/null
os.environ['ANDROID_NDK_HOME'] = f'{SDK}/ndk/25.2.9519653'
print('SDK/NDK готовы')

## Шаг 3. gomobile + Gradle

In [ ]:
import os
!go install golang.org/x/mobile/cmd/gomobile@latest
!go install golang.org/x/mobile/cmd/gobind@latest
!gomobile init
!wget -q https://services.gradle.org/distributions/gradle-8.7-bin.zip -O /tmp/gradle.zip
!unzip -q -o /tmp/gradle.zip -d /opt
os.environ['PATH'] = '/opt/gradle-8.7/bin:' + os.environ['PATH']
!gradle -v

## Шаг 4. Склонировать репозиторий
Замени `REPO_URL` и `BRANCH` на свои.

In [ ]:
REPO_URL = 'https://github.com/USER/REPO.git'   # ← измени
BRANCH   = 'test1'                              # ← измени
!rm -rf /content/repo
!git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo/android_native
!ls

## Шаг 5. Go-ядро → olegcore.aar
Тут компилируется gvisor — самый долгий шаг (несколько минут).

In [ ]:
%cd /content/repo/android_native/gocore
!go mod tidy
!mkdir -p ../app/app/libs
!gomobile bind -target=android -androidapi 21 -o ../app/app/libs/olegcore.aar .
!ls -la ../app/app/libs

## Шаг 6. Сборка APK

In [ ]:
%cd /content/repo/android_native/app
!gradle assembleDebug --no-daemon --stacktrace
!ls -la app/build/outputs/apk/debug

## Шаг 7. Скачать APK

In [ ]:
from google.colab import files
files.download('/content/repo/android_native/app/app/build/outputs/apk/debug/app-debug.apk')